In [1]:
from  datetime import datetime, timedelta
import gc
import numpy as np, pandas as pd
import lightgbm as lgb

In [2]:
#Định nghĩa dữ liệu cho từng cột phân loại
CAL_DTYPES={"event_name_1": "category", "event_name_2": "category", "event_type_1": "category", 
         "event_type_2": "category", "weekday": "category", 'wm_yr_wk': 'int16', "wday": "int16",
        "month": "int16", "year": "int16", "snap_CA": "float32", 'snap_TX': 'float32', 'snap_WI': 'float32' }

#Định nghĩa dữ liệu cho câc cột lien quan đến giá
PRICE_DTYPES = {"store_id": "category", "item_id": "category", "wm_yr_wk": "int16","sell_price":"float32" }

In [3]:
#HiểN thị max 50 dòng
pd.options.display.max_columns = 50

In [4]:
#Định nghĩa một số biến đầu vào
h = 28 
max_lags = 57
tr_last = 1913
fday = datetime(2016,4, 25) 
fday

datetime.datetime(2016, 4, 25, 0, 0)

In [5]:
def create_dt(is_train = True, nrows = None, first_day = 1200):
    prices = pd.read_csv("../dataset/raw/sell_prices.csv", dtype = PRICE_DTYPES)
    for col, col_dtype in PRICE_DTYPES.items():
        if col_dtype == "category":
            prices[col] = prices[col].cat.codes.astype("int16")
            prices[col] -= prices[col].min()
            
    cal = pd.read_csv("../dataset/raw/calendar.csv", dtype = CAL_DTYPES)
    cal["date"] = pd.to_datetime(cal["date"])
    for col, col_dtype in CAL_DTYPES.items():
        if col_dtype == "category":
            cal[col] = cal[col].cat.codes.astype("int16")
            cal[col] -= cal[col].min()
    
    start_day = max(1 if is_train  else tr_last-max_lags, first_day)
    numcols = [f"d_{day}" for day in range(start_day,tr_last+1)]
    catcols = ['id', 'item_id', 'dept_id','store_id', 'cat_id', 'state_id']
    dtype = {numcol:"float32" for numcol in numcols} 
    dtype.update({col: "category" for col in catcols if col != "id"})
    dt = pd.read_csv("../dataset/raw/sales_train_validation.csv", 
                     nrows = nrows, usecols = catcols + numcols, dtype = dtype)
    
    for col in catcols:
        if col != "id":
            dt[col] = dt[col].cat.codes.astype("int16")
            dt[col] -= dt[col].min()
    
    if not is_train:
        for day in range(tr_last+1, tr_last+ 28 +1):
            dt[f"d_{day}"] = np.nan
    
    dt = pd.melt(dt,
                  id_vars = catcols,
                  value_vars = [col for col in dt.columns if col.startswith("d_")],
                  var_name = "d",
                  value_name = "sales")
    
    dt = dt.merge(cal, on= "d", copy = False)
    dt = dt.merge(prices, on = ["store_id", "item_id", "wm_yr_wk"], copy = False)
    
    return dt

In [6]:
def create_fea(dt):
    lags = [7, 28]
    lag_cols = [f"lag_{lag}" for lag in lags ]
    for lag, lag_col in zip(lags, lag_cols):
        dt[lag_col] = dt[["id","sales"]].groupby("id")["sales"].shift(lag)

    wins = [7, 28]
    for win in wins :
        for lag,lag_col in zip(lags, lag_cols):
            dt[f"rmean_{lag}_{win}"] = dt[["id", lag_col]].groupby("id")[lag_col].transform(lambda x : x.rolling(win).mean())

    
    
    date_features = {
        
        "wday": "weekday",
        "week": "isocalendar",
        "month": "month",
        "quarter": "quarter",
        "year": "year",
        "mday": "day",
#         "ime": "is_month_end",
#         "ims": "is_month_start",
    }
    
#     dt.drop(["d", "wm_yr_wk", "weekday"], axis=1, inplace = True)
    
    for date_feat_name, date_feat_func in date_features.items():
        if date_feat_name in dt.columns:
            dt[date_feat_name] = dt[date_feat_name].astype("int16")
        else:
            if date_feat_name == "week":
                dt[date_feat_name] = dt["date"].dt.isocalendar().week.astype("int16")
            else:
                dt[date_feat_name] = getattr(dt["date"].dt, date_feat_func).astype("int16")

In [7]:
FIRST_DAY = 350

In [8]:
#Convert dữ liệu sang dạng long
df = create_dt(is_train=True, first_day= FIRST_DAY, nrows=100)
df.shape

/tmp/ipykernel_95154/771826439.py:38: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(cal, on= "d", copy = False)
/tmp/ipykernel_95154/771826439.py:39: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(prices, on = ["store_id", "item_id", "wm_yr_wk"], copy = False)


(142213, 22)

In [9]:
#HiểN thị một vài dòng đầu
df.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_002_CA_1_validation,1,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,3.97
1,HOBBIES_1_004_CA_1_validation,3,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,4.34
2,HOBBIES_1_005_CA_1_validation,4,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,2.48
3,HOBBIES_1_008_CA_1_validation,7,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,0.50
4,HOBBIES_1_009_CA_1_validation,8,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,1.77


In [10]:
#Làm giàu đặc trưng
create_fea(df)
df.shape

(142213, 31)

In [11]:
#Peview một số dòng đầu
df.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,lag_7,lag_28,rmean_7_7,rmean_28_7,rmean_7_28,rmean_28_28,week,quarter,mday
0,HOBBIES_1_002_CA_1_validation,1,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,3.97,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
1,HOBBIES_1_004_CA_1_validation,3,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,4.34,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
2,HOBBIES_1_005_CA_1_validation,4,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,2.48,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
3,HOBBIES_1_008_CA_1_validation,7,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,0.50,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13
4,HOBBIES_1_009_CA_1_validation,8,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,1.77,NaN,NaN,NaN,NaN,NaN,NaN,2,1,13


In [12]:
#Drop các dòng không có dữ liệU
df.dropna(inplace = True)
df.shape

(136713, 31)

In [13]:
#Các cột phân loại
cat_feats = ['item_id', 'dept_id','store_id', 'cat_id', 'state_id'] + ["event_name_1", "event_name_2", "event_type_1", "event_type_2"]
#Các cột không sử dụng
useless_cols = ["id", "date", "sales","d", "wm_yr_wk", "weekday"]

#Các cột sử dụng trong tập train (ngoại trừ những cột ko dùng ĐN ở trên)
train_cols = df.columns[~df.columns.isin(useless_cols)]

#Tách dữ liệu đặc trưng và mục tiêu trong tập train
X_train = df[train_cols]
y_train = df["sales"]

In [14]:
# train_data = lgb.Dataset(X_train, label = y_train, categorical_feature=cat_feats, free_raw_data=False)
# fake_valid_inds = np.random.choice(len(X_train), 1000000, replace = False)
# fake_valid_data = lgb.Dataset(X_train.iloc[fake_valid_inds], label = y_train.iloc[fake_valid_inds],categorical_feature=cat_feats,
#                              free_raw_data=False)   # This is just a subsample of the training set, not a real validation set !

In [15]:
#Cố định seed random
np.random.seed(777)

#Fake ra 30 000 index cho tập fake valid
fake_valid_inds = np.random.choice(X_train.index.values, 30000, replace = False)

#Index còn lại làm cho tập train
train_inds = np.setdiff1d(X_train.index.values, fake_valid_inds)

#Tập dữ liệu train
train_data = lgb.Dataset(X_train.loc[train_inds] , label = y_train.loc[train_inds], 
                         categorical_feature=cat_feats, free_raw_data=False)

#Tập dữ liệU valid fake
fake_valid_data = lgb.Dataset(X_train.loc[fake_valid_inds], label = y_train.loc[fake_valid_inds],
                              categorical_feature=cat_feats,
                 free_raw_data=False)

In [16]:
#Giải phóng bộ nhớ
del df, X_train, y_train, fake_valid_inds,train_inds ; gc.collect()

0

In [17]:
#Định nghĩa param cho thuật toán LightGBM
params = {
        "objective" : "poisson",
        "metric" :"rmse",
        "force_row_wise" : True,
        "learning_rate" : 0.075,
#         "sub_feature" : 0.8,
        "sub_row" : 0.75,
        "bagging_freq" : 1,
        "lambda_l2" : 0.1,
#         "nthread" : 4
        "metric": ["rmse"],
    'verbosity': 1,
    'num_iterations' : 1200,
    'num_leaves': 128,
    "min_data_in_leaf": 100,
}

In [18]:
m_lgb = lgb.train(params, train_data, valid_sets = [fake_valid_data],  callbacks=[lgb.log_evaluation(20)]) 

[LightGBM] [Info] Total Bins 1313
[LightGBM] [Info] Number of data points in the train set: 106713, number of used features: 21
[LightGBM] [Info] Start training from score 0.186198
[20]	valid_0's rmse: 2.78349
[40]	valid_0's rmse: 2.69029
[60]	valid_0's rmse: 2.67925
[80]	valid_0's rmse: 2.68378
[100]	valid_0's rmse: 2.69012
[120]	valid_0's rmse: 2.69782
[140]	valid_0's rmse: 2.70468
[160]	valid_0's rmse: 2.71064
[180]	valid_0's rmse: 2.7147
[200]	valid_0's rmse: 2.71965
[220]	valid_0's rmse: 2.72517
[240]	valid_0's rmse: 2.72693
[260]	valid_0's rmse: 2.7317
[280]	valid_0's rmse: 2.73572
[300]	valid_0's rmse: 2.73998
[320]	valid_0's rmse: 2.74176
[340]	valid_0's rmse: 2.74453
[360]	valid_0's rmse: 2.74686
[380]	valid_0's rmse: 2.74747
[400]	valid_0's rmse: 2.74857
[420]	valid_0's rmse: 2.75072
[440]	valid_0's rmse: 2.75195
[460]	valid_0's rmse: 2.75353
[480]	valid_0's rmse: 2.75378
[500]	valid_0's rmse: 2.75516
[520]	valid_0's rmse: 2.75641
[540]	valid_0's rmse: 2.7581
[560]	valid_0's 

In [19]:
m_lgb.save_model("model.lgb")

In [20]:

alphas = [1.028, 1.023, 1.018]
weights = [1/len(alphas)]*len(alphas)
sub = 0.

for icount, (alpha, weight) in enumerate(zip(alphas, weights)):

    te = create_dt(False)
    cols = [f"F{i}" for i in range(1,29)]

    for tdelta in range(0, 28):
        day = fday + timedelta(days=tdelta)
        print(tdelta, day)
        tst = te[(te.date >= day - timedelta(days=max_lags)) & (te.date <= day)].copy()
        create_fea(tst)
        tst = tst.loc[tst.date == day , train_cols]
        te.loc[te.date == day, "sales"] = alpha*m_lgb.predict(tst) # magic multiplier by kyakovlev



    te_sub = te.loc[te.date >= fday, ["id", "sales"]].copy()
#     te_sub.loc[te.date >= fday+ timedelta(days=h), "id"] = te_sub.loc[te.date >= fday+timedelta(days=h), 
#                                                                           "id"].str.replace("validation$", "evaluation")
    te_sub["F"] = [f"F{rank}" for rank in te_sub.groupby("id")["id"].cumcount()+1]
    te_sub = te_sub.set_index(["id", "F" ]).unstack()["sales"][cols].reset_index()
    te_sub.fillna(0., inplace = True)
    te_sub.sort_values("id", inplace = True)
    te_sub.reset_index(drop=True, inplace = True)
    te_sub.to_csv(f"submission_{icount}.csv",index=False)
    if icount == 0 :
        sub = te_sub
        sub[cols] *= weight
    else:
        sub[cols] += te_sub[cols]*weight
    print(icount, alpha, weight)


sub2 = sub.copy()
sub2["id"] = sub2["id"].str.replace("validation$", "evaluation")
sub = pd.concat([sub, sub2], axis=0, sort=False)
sub.to_csv("submission.csv",index=False)

/tmp/ipykernel_95154/771826439.py:38: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(cal, on= "d", copy = False)
/tmp/ipykernel_95154/771826439.py:39: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(prices, on = ["store_id", "item_id", "wm_yr_wk"], copy = False)


0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2016-05-10 00:00:00
16 2016-05-11 00:00:00
17 2016-05-12 00:00:00
18 2016-05-13 00:00:00
19 2016-05-14 00:00:00
20 2016-05-15 00:00:00
21 2016-05-16 00:00:00
22 2016-05-17 00:00:00
23 2016-05-18 00:00:00
24 2016-05-19 00:00:00
25 2016-05-20 00:00:00
26 2016-05-21 00:00:00
27 2016-05-22 00:00:00
0 1.028 0.3333333333333333


/tmp/ipykernel_95154/771826439.py:38: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(cal, on= "d", copy = False)
/tmp/ipykernel_95154/771826439.py:39: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(prices, on = ["store_id", "item_id", "wm_yr_wk"], copy = False)


0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2016-05-10 00:00:00
16 2016-05-11 00:00:00
17 2016-05-12 00:00:00
18 2016-05-13 00:00:00
19 2016-05-14 00:00:00
20 2016-05-15 00:00:00
21 2016-05-16 00:00:00
22 2016-05-17 00:00:00
23 2016-05-18 00:00:00
24 2016-05-19 00:00:00
25 2016-05-20 00:00:00
26 2016-05-21 00:00:00
27 2016-05-22 00:00:00
1 1.023 0.3333333333333333


/tmp/ipykernel_95154/771826439.py:38: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(cal, on= "d", copy = False)
/tmp/ipykernel_95154/771826439.py:39: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  dt = dt.merge(prices, on = ["store_id", "item_id", "wm_yr_wk"], copy = False)


0 2016-04-25 00:00:00
1 2016-04-26 00:00:00
2 2016-04-27 00:00:00
3 2016-04-28 00:00:00
4 2016-04-29 00:00:00
5 2016-04-30 00:00:00
6 2016-05-01 00:00:00
7 2016-05-02 00:00:00
8 2016-05-03 00:00:00
9 2016-05-04 00:00:00
10 2016-05-05 00:00:00
11 2016-05-06 00:00:00
12 2016-05-07 00:00:00
13 2016-05-08 00:00:00
14 2016-05-09 00:00:00
15 2016-05-10 00:00:00
16 2016-05-11 00:00:00
17 2016-05-12 00:00:00
18 2016-05-13 00:00:00
19 2016-05-14 00:00:00
20 2016-05-15 00:00:00
21 2016-05-16 00:00:00
22 2016-05-17 00:00:00
23 2016-05-18 00:00:00
24 2016-05-19 00:00:00
25 2016-05-20 00:00:00
26 2016-05-21 00:00:00
27 2016-05-22 00:00:00
2 1.018 0.3333333333333333


In [22]:
sub.head(10)

F,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,FOODS_1_001_CA_1_validation,0.782120,0.634034,0.614782,0.634946,0.784101,0.994203,1.669888,1.043019,1.067495,0.931767,0.546599,1.025106,1.342456,1.055559,0.843035,0.796361,0.812068,0.803196,0.993540,0.941815,1.023715,0.688884,0.690577,0.690038,0.698284,0.788241,0.848211,0.849332
1,FOODS_1_001_CA_2_validation,0.843398,0.693834,0.643314,0.701174,0.897731,1.096447,1.456100,1.133599,1.278471,0.931628,0.625863,1.174417,1.460278,0.920299,1.086076,0.972196,0.893189,0.939709,1.228750,1.367907,1.122580,1.049788,0.858822,0.859673,0.814134,0.840694,1.190934,0.930545
2,FOODS_1_001_CA_3_validation,0.920021,0.767779,0.683606,0.701783,0.834181,1.355129,1.112805,1.124689,1.279849,1.424311,0.564058,1.359946,1.180719,1.029403,1.038868,0.897683,0.928266,0.954907,1.147275,1.266551,0.990202,0.864339,0.792778,0.707624,0.755249,0.865798,0.997098,1.072622
3,FOODS_1_001_CA_4_validation,0.418339,0.295796,0.334804,0.348315,0.410719,0.585521,0.920664,0.627366,0.628886,0.692148,0.282561,0.717022,0.897975,0.636721,0.631690,0.510165,0.576837,0.625325,0.735497,0.849426,0.823834,0.459090,0.502740,0.482390,0.495681,0.639960,0.629796,0.668970
4,FOODS_1_001_TX_1_validation,0.350547,0.312847,0.271900,0.264614,0.220901,0.226099,0.280617,0.306461,0.457326,0.423026,0.211583,0.458904,0.703210,0.521314,0.602017,0.535266,0.576042,0.563580,0.685241,0.642811,0.716287,0.549738,0.516640,0.494863,0.518291,0.599559,0.688211,0.771887
5,FOODS_1_001_TX_2_validation,0.534350,0.444548,0.418507,0.401587,0.334422,0.796496,0.844279,0.648880,0.642740,0.655178,0.346458,0.840302,1.098617,0.767097,0.659721,0.507935,0.559717,0.569582,0.499890,0.685854,0.728995,0.496360,0.440656,0.424799,0.457377,0.647740,0.728604,0.826855
6,FOODS_1_001_TX_3_validation,0.470244,0.429814,0.411434,0.425123,0.570206,0.803096,1.143399,0.614475,0.610448,0.600232,0.342631,0.659629,0.921092,0.714166,0.561875,0.452397,0.490823,0.502505,0.636329,0.704952,0.779050,0.483258,0.426009,0.393535,0.472591,0.570559,0.671313,0.805785
7,FOODS_1_001_WI_1_validation,0.368391,0.420414,0.398238,0.396285,0.397377,0.805049,1.368943,0.768496,0.768035,0.801156,0.533467,1.037597,1.067198,0.717571,0.664947,0.550994,0.617100,0.662939,0.910869,0.922565,0.877290,0.708886,0.592528,0.628437,0.650533,0.795171,0.582302,0.689362
8,FOODS_1_001_WI_2_validation,0.511379,0.402590,0.401693,0.406319,0.432361,0.611190,0.895997,0.670576,0.659516,0.511529,0.316727,0.681831,0.936770,0.677709,0.538250,0.422055,0.456114,0.475293,0.561775,0.669931,0.744423,0.507664,0.454887,0.507709,0.505719,0.599080,0.730056,0.799044
9,FOODS_1_001_WI_3_validation,0.346221,0.294618,0.308389,0.309528,0.287043,0.667066,0.790801,0.630690,0.631537,0.724847,0.327835,0.837420,0.980242,0.743168,0.610312,0.476105,0.545401,0.578378,0.687876,0.777978,0.804244,0.522102,0.504923,0.490713,0.518168,0.671655,0.606372,0.696089
